In [15]:
from IPython.display import display
import pandas as pd
from pathlib import Path

In [16]:

# Dataset Meta data

datasets = {
    "tank": {
        "name":"tank",
        "source":"tanks and temples",
        "resolution": {"width":4300,
                                "height":2189
                                },
        "Points at initialization":206583
    },
    "temple" :{
        "name":"temple",
        "source":"tanks and temples",
        "resolution": {"width":1954,
                                "height":1090
                                },
        "Points at initialization":115017
    },
    "ballroom" :{
        "name":"ballroom",
        "source":"mip-nerf360",
        "resolution": {"width":1957,
                                "height":1091
                                },
        "Points at initialization":207008
    }
}


In [17]:


root = Path("custom-pipeline/Benchmarking/GPU_logs")

def gpu_csv(df:pd.DataFrame,header:str,error):
    prefix = ("cudaoom_","exceeded_")
    header = header.removeprefix(prefix[0])
    header = header.removeprefix(prefix[1])
    stripped = header.split("_")
    dataset_metadata = datasets[stripped[0].casefold()]
    graphics_card = stripped[1].upper()
    vram = str(int(stripped[2].replace("G",""))*1024)+" MiB"
    time_taken = df.Time_Minutes.max()
    resolution = (1/int(stripped[3]))
    
    print(f"""
          Dataset Name: {dataset_metadata["name"]}
          GPU: {graphics_card}
          Video Memory: {vram}
          Time Taken: {time_taken}
          Resolution: {resolution} of {dataset_metadata["resolution"]["width"]} x {dataset_metadata["resolution"]["height"]} ({int(dataset_metadata["resolution"]["width"]*resolution)} x {int(dataset_metadata["resolution"]["height"]*resolution)})
          Error: {error}
          """)

def results_csv(df,header:str):
    stripped = header.split("_")
    del stripped[0]
    
    print(f"{stripped[0].title()} {stripped[1].upper()} x{1/int(stripped[3])}")
    display(df.sort_values("Iterations"))

def router(csv_path:Path):
    df = pd.read_csv(csv_path)
    results_shape = (6,4)
    if df.shape != results_shape:
        error = csv_path.stem.split("_")[0]
        error = error if error in ("cudaoom".casefold(),"exceeded".casefold()) else "No error"
        gpu_csv(df,csv_path.stem,error)
    else:
        results_csv(df,csv_path.stem)
    
def get_data(dataset_name:str):
    csvs = pd.DataFrame([file.name for file in root.glob(f"*{dataset_name.casefold()}*.csv")])
    display(csvs)


In [21]:
get_data("ballroom_rtx3060")
# router(root / "temple_rtx3060_12G_2.csv")
# router(root / "results_temple_rtx3060_12G_2.csv")

,0
0,ballroom_rtx3060_12G_2.csv
1,ballroom_rtx3060_12G_4.csv
2,ballroom_rtx3060_12G_8.csv
3,exceeded_ballroom_rtx3060_12G_1.csv
